In [ ]:
import csv

data = [
    ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same', 'Yes'],
    ['Sunny', 'Warm', 'High', 'Strong', 'Warm', 'Same', 'Yes'],
    ['Rainy', 'Cold', 'High', 'Strong', 'Warm', 'Change', 'No'],
    ['Sunny', 'Warm', 'High', 'Strong', 'Cool', 'Change', 'Yes']
]

num_attributes = len(data[0]) - 1
hypothesis = ['Ø'] * num_attributes

print("Initial Hypothesis:", hypothesis)

for example in data:
    if example[-1] == 'Yes':
        for i in range(num_attributes):
            if hypothesis[i] == 'Ø':
                hypothesis[i] = example[i]
            elif hypothesis[i] != example[i]:
                hypothesis[i] = '?'

        print("Updated Hypothesis:", hypothesis)

print("Final Hypothesis:", hypothesis)

Initial Hypothesis: ['Ø', 'Ø', 'Ø', 'Ø', 'Ø', 'Ø']
Updated Hypothesis: ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same']
Updated Hypothesis: ['Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same']
Updated Hypothesis: ['Sunny', 'Warm', '?', 'Strong', '?', '?']
Final Hypothesis: ['Sunny', 'Warm', '?', 'Strong', '?', '?']


In [ ]:
import csv
with open('/content/trainingdata.csv') as f:
    data = list(csv.reader(f))
examples = data[1:]
n = len(examples[0]) - 1

S = ['Ø'] * n
G = [['?'] * n]
def consistent(h, x):
    return all(h[i] == '?' or h[i] == x[i] for i in range(len(h)))
for x in examples:
    if x[-1] == 'Yes':
        G = [g for g in G if consistent(g, x)]
        for i in range(n):
            if S[i] == 'Ø':
                S[i] = x[i]
            elif S[i] != x[i]:
                S[i] = '?'
    else:
        new_G = []
        for g in G:
            if consistent(g, x):
                for i in range(n):
                    if g[i] == '?':
                        h = g.copy()
                        h[i] = S[i]
                        new_G.append(h)
            else:
                new_G.append(g)
        G = new_G
print("Final Specific Boundary S:", S)
print("Final General Boundary G:", G)

Final Specific Boundary S: ['Sunny', 'Warm', '?', 'Strong', '?', '?']
Final General Boundary G: [['Sunny', '?', '?', '?', '?', '?'], ['?', 'Warm', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', '?'], ['?', '?', '?', 'Strong', '?', '?']]


In [4]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier


data = {
    'Outlook': ['Sunny','Sunny','Overcast','Rain','Rain','Rain','Overcast',
                'Sunny','Sunny','Rain','Sunny','Overcast','Overcast','Rain'],
    'Humidity': ['High','High','High','High','Normal','Normal','Normal',
                 'High','Normal','Normal','Normal','High','Normal','High'],
    'Wind': ['Weak','Strong','Weak','Weak','Weak','Strong','Strong',
             'Weak','Weak','Weak','Strong','Strong','Weak','Strong'],
    'Play': ['No','No','Yes','Yes','Yes','No','Yes',
             'No','Yes','Yes','Yes','Yes','Yes','No']
}

df = pd.DataFrame(data)


X = pd.get_dummies(df.drop('Play', axis=1))
y = df['Play']


model = DecisionTreeClassifier(criterion='entropy')
model.fit(X, y)


sample = pd.DataFrame({
    'Outlook': ['Sunny'],
    'Humidity': ['High'],
    'Wind': ['Strong']
})

sample_encoded = pd.get_dummies(sample)
sample_encoded = sample_encoded.reindex(columns=X.columns, fill_value=0)

prediction = model.predict(sample_encoded)

print(prediction[0])

No


In [5]:
import pandas as pd
import numpy as np
from math import log2


data = {
    'Outlook': ['Sunny','Sunny','Overcast','Rain','Rain','Overcast'],
    'Humidity': ['High','High','High','Normal','Normal','Normal'],
    'Wind': ['Weak','Strong','Weak','Weak','Strong','Strong'],
    'Play': ['No','No','Yes','Yes','No','Yes']
}

df = pd.DataFrame(data)


def entropy(target_col):
    values, counts = np.unique(target_col, return_counts=True)
    entropy_value = 0
    for i in range(len(values)):
        p = counts[i] / np.sum(counts)
        entropy_value -= p * log2(p)
    return entropy_value

def information_gain(data, split_attr, target_attr):
    total_entropy = entropy(data[target_attr])
    values, counts = np.unique(data[split_attr], return_counts=True)

    weighted_entropy = 0
    for i in range(len(values)):
        subset = data[data[split_attr] == values[i]]
        weighted_entropy += (counts[i] / np.sum(counts)) * entropy(subset[target_attr])

    return total_entropy - weighted_entropy


attributes = ['Outlook', 'Humidity', 'Wind']
gains = {}

for attr in attributes:
    gains[attr] = information_gain(df, attr, 'Play')

best_attr = max(gains, key=gains.get)

print("Information Gain Values:")
for k, v in gains.items():
    print(k, ":", round(v, 4))

print("\nBest Attribute to Split (Heuristic Choice):", best_attr)

Information Gain Values:
Outlook : 0.6667
Humidity : 0.0817
Wind : 0.0817

Best Attribute to Split (Heuristic Choice): Outlook


In [6]:
from sklearn.datasets import load_iris
from sklearn.linear_model import Perceptron
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

iris = load_iris()
X = iris.data
y = iris.target
y = (y != 0).astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)
model = Perceptron()
model.fit(X_train, y_train)
predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

accuracy

1.0

In [7]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

iris = load_iris()
X = iris.data
y = iris.target

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)


model = MLPClassifier(hidden_layer_sizes=(10,),
                      activation='relu',
                      solver='adam',
                      max_iter=1000,
                      random_state=42)


model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 1.0

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        13
           2       1.00      1.00      1.00        13

    accuracy                           1.00        45
   macro avg       1.00      1.00      1.00        45
weighted avg       1.00      1.00      1.00        45



/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


In [1]:
import numpy as np
from gplearn.genetic import SymbolicRegressor
from sklearn.metrics import mean_squared_error


np.random.seed(0)
X = np.linspace(-5, 5, 100).reshape(-1, 1)
y = X[:, 0]**2 + X[:, 0] + 1


model = SymbolicRegressor(
    population_size=500,
    generations=20,
    tournament_size=20,
    stopping_criteria=0.01,
    const_range=(-5, 5),
    random_state=0
)

model.fit(X, y)

y_pred = model.predict(X)

print("Best Expression:", model._program)
print("MSE:", mean_squared_error(y, y_pred))

Best Expression: add(X0, add(mul(X0, X0), div(X0, X0)))
MSE: 1.910522504832138e-32


In [2]:
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score


iris = load_iris()
X = iris.data
y = iris.target


model = DecisionTreeClassifier(criterion='entropy')

scores = cross_val_score(model, X, y, cv=5)

print("Fold Accuracies:", scores)
print("Mean Accuracy:", scores.mean())

Fold Accuracies: [0.96666667 0.96666667 0.9        0.96666667 1.        ]
Mean Accuracy: 0.9600000000000002


In [3]:
import numpy as np


def h1(x): return "Yes"
def h2(x): return "No"
def h3(x): return "Yes" if x == 1 else "No"
def h4(x): return "Yes" if x == 0 else "No"

hypotheses = [h1, h2, h3, h4]
hyp_names = ["h1", "h2", "h3", "h4"]


posterior = np.array([0.25, 0.25, 0.25, 0.25])


training_data = [(1, "Yes"), (0, "No")]

for x, label in training_data:
    likelihood = []

    for h in hypotheses:
        prediction = h(x)
        if prediction == label:
            likelihood.append(1)
        else:
            likelihood.append(0)

    likelihood = np.array(likelihood)


    posterior = likelihood * posterior
    if posterior.sum() != 0:
        posterior = posterior / posterior.sum()

print(dict(zip(hyp_names, posterior)))

{'h1': np.float64(0.0), 'h2': np.float64(0.0), 'h3': np.float64(1.0), 'h4': np.float64(0.0)}
